In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nusratjahannawal/banglapsg-review-sheet/BanglaPSG_Review_Sheet.xlsx


In [2]:
import os, glob, re, unicodedata
import numpy as np, pandas as pd

hits = glob.glob("/kaggle/input/datasets/nusratjahannawal/banglapsg-review-sheet/BanglaPSG_Review_Sheet.xlsx", recursive=True)
print("Found:", hits)
assert len(hits) == 1, "File not found or multiple matches; set F manually from the list above"
F = hits[0]
OUT = "/kaggle/working"
VALID_SEV = {"Low", "Medium", "High"}

rev  = pd.read_excel(F, sheet_name="review")
base = pd.read_excel(F, sheet_name="base_clean")
cmap = pd.read_excel(F, sheet_name="cluster_map")
fz   = pd.read_excel(F, sheet_name="fuzzy_pairs")
VALID_CAT = set(base["Category"])
print(base.shape, "| clusters:", cmap["Cluster_ID"].nunique(), "| review clusters:", rev["Cluster_ID"].nunique())

Found: ['/kaggle/input/datasets/nusratjahannawal/banglapsg-review-sheet/BanglaPSG_Review_Sheet.xlsx']
(6105, 5) | clusters: 129 | review clusters: 64


In [3]:
rev = rev.replace(r"^\s*$", np.nan, regex=True)

manual_ids = set(cmap.loc[cmap["Resolution"] == "manual", "Cluster_ID"])
dec = rev[rev["Is_Duplicate"].notna()].copy()
dec["Is_Duplicate"] = dec["Is_Duplicate"].astype(str).str.strip().str.upper()

assert dec["Cluster_ID"].is_unique, "A cluster has more than one decision row"
missing = manual_ids - set(dec["Cluster_ID"])
assert not missing, f"{len(missing)} clusters have no decision: {sorted(missing)[:10]}"
assert set(dec["Is_Duplicate"]) <= {"Y", "N"}, "Is_Duplicate must be Y or N"

members = cmap.groupby("Cluster_ID")["Original_No"].apply(sorted).to_dict()
for _, r in dec[dec["Is_Duplicate"] == "Y"].iterrows():
    cid = r["Cluster_ID"]
    assert str(r["Final_Category"]).strip() in VALID_CAT, f"{cid}: Final_Category is invalid or empty"
    assert str(r["Final_Severity"]).strip() in VALID_SEV, f"{cid}: Final_Severity is invalid or empty"
    if pd.notna(r["Keep_Original_No"]):
        assert int(r["Keep_Original_No"]) in members[cid], f"{cid}: Keep_Original_No is outside this cluster"
print("Decisions OK:", len(dec), "| Y:", (dec["Is_Duplicate"] == "Y").sum(),
      "| N:", (dec["Is_Duplicate"] == "N").sum())

Decisions OK: 64 | Y: 64 | N: 0


In [4]:
res  = cmap.drop_duplicates("Cluster_ID").set_index("Cluster_ID")["Resolution"].to_dict()
ctyp = cmap.drop_duplicates("Cluster_ID").set_index("Cluster_ID")["type"].to_dict()
decd = dec.set_index("Cluster_ID")

final = base.set_index("Original_No").copy()
removed = []

for cid, mem in members.items():
    if res[cid] == "auto":
        keep, reason = mem[0], f"auto ({ctyp[cid]}, labels identical)"
    else:
        d = decd.loc[cid]
        if d["Is_Duplicate"] == "N":
            continue
        keep = int(d["Keep_Original_No"]) if pd.notna(d["Keep_Original_No"]) else mem[0]
        final.loc[keep, "Category"] = str(d["Final_Category"]).strip()
        final.loc[keep, "Severity"] = str(d["Final_Severity"]).strip()
        reason = f"manual ({ctyp[cid]}): " + ("" if pd.isna(d["Reason"]) else str(d["Reason"]))
    for m in mem:
        if m != keep:
            removed.append(dict(Original_No=m, Kept_Original_No=keep, Cluster_ID=cid, Reason=reason))

removed = pd.DataFrame(removed).sort_values("Original_No").reset_index(drop=True)
final = final.drop(index=removed["Original_No"]).sort_index().reset_index()
print("Before:", len(base), "| Removed:", len(removed), "| After:", len(final))

Before: 6105 | Removed: 135 | After: 5970


In [5]:
def norm_key(t):
    t = unicodedata.normalize("NFC", str(t))
    t = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", t)
    t = t.replace("\\n", " ")
    t = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", t)
    return re.sub(r"\s+", "", re.sub(r"\s+", " ", t).strip().lower())

assert len(base) - len(removed) == len(final), "Row count mismatch"
assert final["Original_No"].is_unique
assert final[["Text", "Category", "Severity"]].notna().all().all()
assert set(final["Severity"]) <= VALID_SEV and set(final["Category"]) <= VALID_CAT
assert final["Text"].duplicated().sum() == 0, "Exact duplicate text remains (check clusters marked N)"
assert final["Text"].map(norm_key).duplicated().sum() == 0, "Normalized duplicate remains"

left = set(final["Original_No"])
still = fz[fz["No_A"].isin(left) & fz["No_B"].isin(left)]
print("Fuzzy pairs still present (should all be your N decisions):", len(still))
print("\nALL CHECKS PASSED")

Fuzzy pairs still present (should all be your N decisions): 0

ALL CHECKS PASSED


In [6]:
final = final.reset_index(drop=True)
final["No."] = range(1, len(final) + 1)

clean = final[["No.", "Text", "Category", "Severity"]].rename(columns={"Text": "Grievance Text"})
clean.to_excel(f"{OUT}/BanglaPSG_V2_Deduplicated.xlsx", index=False)

idmap = final[["No.", "Original_No", "Source_No"]].rename(columns={"No.": "New_No"})
summary = pd.DataFrame({
    "Metric": ["Rows before", "Rows removed", "Rows after",
               "Auto-resolved clusters", "Manual clusters", "Manual Y", "Manual N"],
    "Value": [len(base), len(removed), len(final),
              sum(v == "auto" for v in res.values()), len(manual_ids),
              int((dec["Is_Duplicate"] == "Y").sum()), int((dec["Is_Duplicate"] == "N").sum())]})

with pd.ExcelWriter(f"{OUT}/BanglaPSG_Dedup_Audit.xlsx") as w:
    removed.to_excel(w, sheet_name="removal_log", index=False)
    idmap.to_excel(w, sheet_name="id_mapping", index=False)
    dec.to_excel(w, sheet_name="adjudication", index=False)
    summary.to_excel(w, sheet_name="summary", index=False)

print(summary.to_string(index=False))
print("\nSeverity before:\n", base["Severity"].value_counts(), "\nSeverity after:\n", clean["Severity"].value_counts())
print("\nCategory before:\n", base["Category"].value_counts(), "\nCategory after:\n", clean["Category"].value_counts())

                Metric  Value
           Rows before   6105
          Rows removed    135
            Rows after   5970
Auto-resolved clusters     65
       Manual clusters     64
              Manual Y     64
              Manual N      0

Severity before:
 Severity
Medium    2464
Low       1988
High      1653
Name: count, dtype: int64 
Severity after:
 Severity
Medium    2410
Low       1951
High      1609
Name: count, dtype: int64

Category before:
 Category
Road & Transport    1061
Food                1036
Waste               1015
Healthcare          1015
Electricity         1007
Water                971
Name: count, dtype: int64 
Category after:
 Category
Food                1030
Road & Transport    1025
Electricity          995
Waste                992
Healthcare           983
Water                945
Name: count, dtype: int64


In [8]:
r = removed.copy()
parts = r["Reason"].str.extract(r"^(auto|manual) \((exact|normalized|fuzzy)")
r["Kind"] = parts[0] + " / " + parts[1]
print(r["Kind"].value_counts(dropna=False))
print("Total removed:", len(r))
print("Clusters:", r["Cluster_ID"].nunique())
print(r.groupby("Cluster_ID").size().value_counts())

Kind
auto / exact           57
manual / exact         42
manual / fuzzy         16
manual / normalized    11
auto / normalized       9
Name: count, dtype: int64
Total removed: 135
Clusters: 129
1    123
2      6
Name: count, dtype: int64
